# 🟩 Explicación general del código en español:
Este script está diseñado para hacer predicción de etiquetas para oraciones (Claim, Evidence, Other) utilizando un modelo RoBERTa entrenado previamente en Google Colab. A continuación, se describen las principales etapas:

1. 📦 Instalación e importación de librerías
Se instalan las librerías necesarias y se importan módulos como transformers, torch, pandas y re.

2. 🚀 Carga del modelo
Se monta Google Drive y se carga un modelo previamente entrenado (checkpoint-336) y su tokenizador, ubicados en una carpeta compartida. El modelo es del tipo AutoModelForSequenceClassification de HuggingFace.

3. 🧠 Función de predicción (predict_with_softmax)
Se define una función que:

Toma entradas de texto (fragmentos con contexto).

Las tokeniza.

Realiza inferencia con el modelo.

Devuelve las probabilidades softmax para cada clase (Claim, Evidence, Other).

4. 🧩 Extracción de bloques Claim–Evidence
Se procesan respuestas largas para extraer bloques explícitos donde el texto tiene estructura tipo Título: Contenido.

Se extraen estos pares.

Se eliminan del texto original para evitar duplicación.

5. 🧾 División en oraciones generales
El texto restante (sin bloques) se divide en oraciones normales usando puntuación (., ?, !).

6. 🧷 Unión de todas las oraciones
Se unen:

Frases Claim y Evidence (de los bloques).

Oraciones generales.

Cada una se marca como "Claim", "Evidence" o "General".

7. 🔁 Generación de contexto
Para cada oración, se genera una ventana de contexto de ±4 oraciones. Esto se usa como input para que el modelo tenga mayor comprensión contextual.

8. 📊 Inferencia con el modelo
Se realiza inferencia para todos los fragmentos con contexto. Se obtiene:

Probabilidad de ser Claim.

Probabilidad de ser Evidence.

Etiqueta predicha según softmax.

9. 🔧 Corrección basada en palabras clave
Para mejorar precisión, se aplican reglas:

Si el modelo predice "Other" pero la oración contiene palabras clave típicas de Claim o Evidence, se cambia la etiqueta.

También se aplica una lógica basada en reglas manuales (Rule_Label).

10. 📈 Análisis de resultados
Se imprime la distribución de etiquetas finales (Final_Label).

11. 💾 Guardado
Los resultados se guardan en un archivo .xlsx para su posterior análisis.

In [ ]:
!pip install transformers

In [ ]:
# ✅ Instalación de librerías necesarias
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.nn.functional import softmax
from pathlib import Path
import os
from google.colab import drive

In [ ]:
# ✅ 2. Montar Google Drive y cargar modelo
# Se carga un modelo previamente entrenado de RoBERTa para clasificación de texto

drive.mount('/content/drive')
model_path = Path("models/claim_evidence_roberta/checkpoint-336")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
# ✅ Función para extraer bloques tipo Claim:Evidence
# Soporta formatos con y sin numeración (ej. '1. Título:' o 'Título:')

def extract_and_split_blocks(text):
    """
    """
    pattern = r'(\d+[\.\-]*\s?[^\n\r:]{1,80}):'
    parts = []
    matches = list(re.finditer(pattern, text))

    last_end = 0
    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        parts.append(block)
        last_end = end

    tail = text[last_end:].strip()
    additional_blocks = [b.strip() for b in re.split(r'\n+', tail) if ':' in b and len(b.split(":", 1)[0].split()) <= 10]
    parts.extend(additional_blocks)

    results = []
    for p in parts:
        if ":" in p:
            claim, evidence = p.split(":", 1)
            results.append((claim.strip(), evidence.strip()))
    return results

In [ ]:
# ✅ Separador general de oraciones basado en puntuación
# (puntos, signos de exclamación e interrogación)

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

In [ ]:
def get_contextual_input(i, sentence_list, window=4):
    left = max(i - window, 0)
    right = min(i + window + 1, len(sentence_list))
    return " ".join(sentence_list[left:right])

In [ ]:
# ✅ Función de predicción por lotes con softmax
# Devuelve probabilidades por clase (Other, Claim, Evidence)

def predict_with_softmax(texts, model, tokenizer, batch_size=16):
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="🔍 Predicting w/ softmax"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.extend(probs)
    return all_probs

In [ ]:
# ✅ Cargar y filtrar los datos
# Solo se consideran 5 postulantes para ejemplo

df = pd.read_excel("data/applications.xlsx")#, encoding="latin1")
selected_ids = df["applicant_id"].drop_duplicates().head(5)
df = df[df["applicant_id"].isin(selected_ids)].copy()

In [ ]:
# ✅ Procesamiento de texto y preparación para predicción contextualizada

rows = []
for _, row in df.iterrows():
    texto = row["answer_text"]
    applicant = row["applicant_id"]
    pregunta = row["question_text"]
    answer_id = f"{applicant}_{pregunta[:30]}"

    # ✅ Extracción de bloques Claim–Evidence

    blocks = extract_and_split_blocks(texto)
    extracted_texts = [f"{c}: {e}" for c, e in blocks]

    # ✅ Eliminación de bloques del texto original (normalización de espacios)

    remaining = texto
    for t in extracted_texts:
        t_norm = re.sub(r'\s+', ' ', t.strip())
        remaining_norm = re.sub(r'\s+', ' ', remaining)
        if t_norm in remaining_norm:
            remaining = remaining_norm.replace(t_norm, '')

    # ✅ División en oraciones generales (las que no son Claim–Evidence)

    general_sentences = split_sentences(remaining)

    # ✅ Combinación de Claim–Evidence + oraciones generales

    merged_sentences = []
    for c, e in blocks:
        merged_sentences.append((c, "Claim"))
        merged_sentences.append((e, "Evidence"))
    for s in general_sentences:
        merged_sentences.append((s, "General"))

    # ✅ Construcción de filas con contexto incluido

    fragments = [m[0] for m in merged_sentences]
    for idx, (frag, rule_label) in enumerate(merged_sentences):
        context_input = get_contextual_input(idx, fragments, window=4)
        rows.append({
            "applicant_id": applicant,
            "question_text": pregunta,
            "Fragment": frag,
            "Context_Input": context_input,
            "Fragment_Index": idx,
            "answer_id": answer_id,
            "Rule_Label": rule_label
        })

df_contextual = pd.DataFrame(rows)

In [ ]:
# ✅ Ejecutar predicción

texts = df_contextual["Context_Input"].tolist()
probs = predict_with_softmax(texts, model, tokenizer)
df_contextual["Claim_Prob"] = [p[1] for p in probs]  # Claim
df_contextual["Evidence_Prob"] = [p[2] for p in probs]  # Evidence
df_contextual["Pred_Label"] = [["Other", "Claim", "Evidence"][int(p.argmax())] for p in probs]

In [ ]:
# ✅ Corrección basada en palabras clave

claim_keywords = [
    "crear", "ofrecer", "desarrollar", "mejorar", "posicionar", "expandir",
    "objetivo", "proveer", "pretendemos", "queremos", "buscamos",
    "entregar", "diseñar", "plan", "esperamos", "aspiramos", "tenemos como meta",
    "planeamos", "implementaremos", "impulsaremos", "estableceremos", "enfocamos en",
    "promovemos", "proponemos", "queremos lograr", "nuestro propósito", "nuestra meta",
    "buscamos impactar", "nos comprometemos a", "pretendemos lograr", "visamos", "iniciativa para",
    "daremos inicio", "se plantea", "se prevé"
]
evidence_keywords = [
    "realizó", "participó", "asistió", "implementó", "ejecutó",
    "ventas", "aumento", "crecimiento", "porcentaje", "resultados",
    "acciones", "actividades", "campañas", "estrategias", "logramos",
    "datos demuestran", "se realizó", "hemos colaborado", "se han implementado",
    "actualmente trabajamos", "tenemos convenios", "ya contamos con",
    "se ha desarrollado", "se evidenció", "reporte de", "impacto medido",
    "en primer lugar", "además", "por ejemplo", "por otro lado", "como resultado",
    "se gestionó", "actualmente se encuentra", "tenemos registro de",
    "trabajamos con", "colaboramos con", "contamos con experiencia", "tenemos historial de"
]

df_contextual["Is_Claim_Keyword"] = df_contextual["Fragment"].apply(lambda x: any(k in x.lower() for k in claim_keywords))
df_contextual["Is_Evidence_Keyword"] = df_contextual["Fragment"].apply(lambda x: any(k in x.lower() for k in evidence_keywords))

In [ ]:
# ✅ Umbral + reglas para corrección final de etiquetas

threshold_claim = 0.40
threshold_evidence = 0.40

df_contextual["Final_Label"] = df_contextual["Pred_Label"]

df_contextual.loc[
    (df_contextual["Pred_Label"] == "Other") &
    ((df_contextual["Claim_Prob"] >= threshold_claim) | (df_contextual["Is_Claim_Keyword"])),
    "Final_Label"
] = "Claim"

df_contextual.loc[
    (df_contextual["Final_Label"] == "Other") &
    ((df_contextual["Evidence_Prob"] >= threshold_evidence) | (df_contextual["Is_Evidence_Keyword"])),
    "Final_Label"
] = "Evidence"

# 👉 Rule-based override
# ✅ Etiquetas originales por regla tienen prioridad
# (bloques tipo Claim:Evidence detectados por expresiones regulares)
df_contextual.loc[df_contextual["Rule_Label"] == "Claim", "Final_Label"] = "Claim"
df_contextual.loc[df_contextual["Rule_Label"] == "Evidence", "Final_Label"] = "Evidence"

In [ ]:
# ✅ Visualizar distribuci\u00f3n final
print("📊 Final label distribution:")
print(df_contextual["Final_Label"].value_counts(normalize=True) * 100)

📊 Final label distribution:
Final_Label
Evidence    39.109698
Claim       34.499205
Other       26.391097
Name: proportion, dtype: float64


In [ ]:
# ✅ Guardar resultados
df_contextual.to_excel("data/predicted_claim_evidence.xlsx", index=False)